<a href="https://colab.research.google.com/github/manurudeepika02-del/Infosys_FreightQuote_AI/blob/main/Login_Page.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q streamlit streamlit-option-menu pyngrok pyjwt bcrypt plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 52.6 MB/s eta 0:00:00


In [21]:
%%writefile app.py
import os, sqlite3, jwt, bcrypt, datetime, time, random, smtplib
from email.mime.text import MIMEText
import streamlit as st
import plotly.graph_objects as go
from streamlit_option_menu import option_menu

# ============================================================
# EMAIL CONFIG (fill these in, or set as Colab secrets and read via userdata)
# ============================================================
SMTP_EMAIL = os.environ.get("SMTP_EMAIL", "noreply.infosysspringboard@gmail.com")
SMTP_APP_PASSWORD = os.environ.get("SMTP_APP_PASSWORD", "kgqnncrhcbwuxihm")  # ⚠️ rotate this password since it was shared in chat

def send_otp_email(to_email, otp):
    try:
        msg = MIMEText(f"Your Infosys Freight Quote Portal verification code is: {otp}\n\nThis code expires in 5 minutes.")
        msg["Subject"] = "Infosys Freight Quote Portal - Password Reset OTP"
        msg["From"] = SMTP_EMAIL
        msg["To"] = to_email
        with smtplib.SMTP("smtp.gmail.com", 587) as server:
            server.starttls()
            server.login(SMTP_EMAIL, SMTP_APP_PASSWORD)
            server.sendmail(SMTP_EMAIL, to_email, msg.as_string())
        return True
    except Exception as e:
        st.error(f"Email send failed: {e}")
        return False

# ============================================================
# PAGE / THEME CONFIG — Classic Navy & Gold Professional Theme
# ============================================================
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#c9a24b"\nbackgroundColor="#f4f5f7"\nsecondaryBackgroundColor="#ffffff"\ntextColor="#1b2436"\n')

st.set_page_config(page_title="Infosys Freight Quote Portal", page_icon="🏛️", layout="wide", initial_sidebar_state="expanded")

COLORS = {
    "navy_deep":   "#0b1530",
    "navy":        "#0f1c3f",
    "navy_light":  "#1c2e5c",
    "gold":        "#c9a24b",
    "gold_hover":  "#b8912f",
    "gold_light":  "#e8d9ad",
    "bg_main":     "#f4f5f7",
    "bg_card":     "#ffffff",
    "text_main":   "#1b2436",
    "text_heading":"#0b1530",
    "text_muted":  "#5b6478",
    "text_on_navy":"#f4f5f7",
    "border":      "#d8dbe2",
    "success":     "#2f8f5b",
    "danger":      "#b3413a",
}

JWT_SECRET = "super-secret-infosys-key-2026"

# ============================================================
# CLASSIC PROFESSIONAL CSS
# ============================================================
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700;800&family=Inter:wght@300;400;500;600;700&display=swap');

    html, body, .stApp {{
        background: {COLORS['bg_main']} !important;
        font-family: 'Inter', sans-serif !important;
        color: {COLORS['text_main']} !important;
    }}

    footer, div[data-testid="stDecoration"] {{ visibility: hidden !important; display: none !important; }}
    header {{ background: transparent !important; z-index: 999999 !important; }}

    button[kind="header"], div[data-testid="stSidebarCollapsedControl"] button {{
        visibility: visible !important; display: flex !important; opacity: 1 !important;
        background-color: {COLORS['navy']} !important; border: 1px solid {COLORS['gold']} !important;
        border-radius: 6px !important; padding: 6px !important; margin: 8px !important;
    }}
    button[kind="header"] svg, div[data-testid="stSidebarCollapsedControl"] svg {{
        fill: {COLORS['gold']} !important; color: {COLORS['gold']} !important; stroke: {COLORS['gold']} !important;
    }}

    .block-container {{ padding: 2rem 2.5rem !important; max-width: 1200px; }}

    h1, h2, h3, h4 {{
        font-family: 'Playfair Display', serif !important;
        color: {COLORS['text_heading']} !important;
        letter-spacing: 0.3px;
    }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_heading']} !important; font-size: 13px !important; letter-spacing: 0.2px; }}

    /* Inputs — classic bordered style, forced light background everywhere */
    div[data-baseweb="base-input"], div[data-baseweb="select"] > div {{ background-color: {COLORS['bg_card']} !important; border: none !important; }}
    div[data-baseweb="input"], div[data-baseweb="select"], div[data-baseweb="popover"], div[data-baseweb="menu"], ul[data-baseweb="menu"] {{
        background-color: {COLORS['bg_card']} !important;
        border: 1.5px solid {COLORS['border']} !important;
        border-radius: 6px !important;
    }}
    div[data-baseweb="input"]:focus-within {{
        border-color: {COLORS['gold']} !important;
        box-shadow: 0 0 0 3px rgba(201,162,75,0.18) !important;
    }}
    input, textarea, div[data-baseweb="select"] span, li[role="option"] {{
        color: {COLORS['text_main']} !important; -webkit-text-fill-color: {COLORS['text_main']} !important;
        background-color: {COLORS['bg_card']} !important;
    }}
    li[role="option"]:hover {{ background-color: {COLORS['gold_light']} !important; }}

    /* Buttons — deep navy with gold accent border, classic feel */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['navy']} !important; color: {COLORS['gold_light']} !important;
        border: 1px solid {COLORS['navy']} !important; border-radius: 6px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 600 !important; font-size: 14px !important;
        height: 46px !important; min-height: 46px !important; letter-spacing: 0.4px;
        display: flex !important; align-items: center !important; justify-content: center !important;
        padding: 0px 16px !important; width: 100%; transition: all 0.2s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['gold']} !important; color: {COLORS['navy_deep']} !important;
        border-color: {COLORS['gold']} !important;
    }}

    section[data-testid="stSidebar"] {{
        background: linear-gradient(180deg, {COLORS['navy']} 0%, {COLORS['navy_deep']} 100%) !important;
        border-right: 1px solid {COLORS['gold']} !important;
    }}
    section[data-testid="stSidebar"] * {{ color: {COLORS['text_on_navy']} !important; }}

    .pn-card {{
        background: {COLORS['bg_card']};
        border: 1px solid {COLORS['border']};
        border-top: 3px solid {COLORS['gold']};
        border-radius: 10px;
        padding: 24px;
        box-shadow: 0 2px 10px rgba(11,21,48,0.06);
    }}

    .divider-gold {{ height: 2px; background: linear-gradient(90deg, transparent, {COLORS['gold']}, transparent); margin: 12px 0 20px; }}

    h1.freight-banner-title {{ color: #ffffff !important; }}
</style>
""", unsafe_allow_html=True)

# ============================================================
# DB
# ============================================================
def get_db(): return sqlite3.connect("infosys_portal.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("""CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT UNIQUE, email TEXT UNIQUE,
        password_hash TEXT, security_question TEXT, security_answer_hash TEXT)""")
    if not conn.execute("SELECT id FROM users WHERE email='infosys@ai'").fetchone():
        conn.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)",
                     ("Administrator", "infosys@ai", hash_txt("admin@123"), "What is your pet name?", hash_txt("admin")))

def make_jwt(email): return jwt.encode({"email": email, "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=2)}, JWT_SECRET, algorithm="HS256")
def verify_jwt(token):
    try: return jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except: return None

for k, v in [("token", None), ("page", "Login"), ("reset_email", None), ("reset_mode", None),
             ("otp_code", None), ("otp_expiry", None), ("otp_sent_to", None)]:
    if k not in st.session_state: st.session_state[k] = v

def navigate(p): st.session_state.page = p; st.rerun()

def auth_header(title, sub="Enterprise Intelligence Portal"):
    st.markdown(f"""
    <div style="text-align:center;padding:1.2rem 0 0.6rem;">
        <div style="font-size:34px;margin-bottom:6px;">🏛️</div>
        <h1 style="font-size:2rem !important;margin:0;">Infosys Freight Quote Portal</h1>
        <p style="color:{COLORS['text_muted']};font-size:13px;margin:4px 0 0;letter-spacing:0.5px;">{sub}</p>
    </div>
    <div class="divider-gold" style="max-width:120px;margin-left:auto;margin-right:auto;"></div>
    <div style="text-align:center;margin-bottom:1.2rem;"><span style="font-size:1.05rem;font-weight:700;color:{COLORS['text_heading']};">{title}</span></div>
    """, unsafe_allow_html=True)

# ============================================================
# PAGE ROUTING
# ============================================================
if not st.session_state.token:
    if st.session_state.page not in ["Login", "Signup", "Forgot"]:
        st.session_state.page = "Login"

    _, mid, _ = st.columns([1, 1.45, 1])
    with mid:
        if st.session_state.page == "Login":
            auth_header("Sign in to your account")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="••••••••")
            st.markdown("<br>", unsafe_allow_html=True)

            col_l, col_c, col_r = st.columns([1, 1.15, 1.3])
            if col_l.button("Sign In →", use_container_width=True):
                with get_db() as c: r = c.execute("SELECT password_hash FROM users WHERE email=?", (email,)).fetchone()
                if r and check_txt(pwd, r[0]): st.session_state.token = make_jwt(email); navigate("Dashboard")
                else: st.error("❌ Invalid credentials.")
            if col_c.button("Create Account", use_container_width=True): navigate("Signup")
            if col_r.button("Forgot Password", use_container_width=True): navigate("Forgot")

        elif st.session_state.page == "Signup":
            auth_header("Create an account", "Join Infosys Freight Quote Portal today")
            uname = st.text_input("Full name / Username", placeholder="Jane Doe")
            email = st.text_input("Email address", placeholder="you@infosys.com").lower().strip()
            pwd = st.text_input("Password", type="password", placeholder="Min. 8 characters")
            confirm_pwd = st.text_input("Confirm password", type="password", placeholder="Re-enter password")
            sq = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favourite city?"])
            sa = st.text_input("Your answer", placeholder="Security answer")
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Create Account & Login →", use_container_width=True):
                if not uname.strip():
                    st.error("⚠️ Username is required.")
                elif not email.strip():
                    st.error("⚠️ Email is required.")
                elif len(pwd) < 8:
                    st.error(f"⚠️ Password must be at least 8 characters (yours is {len(pwd)}).")
                elif not sa.strip():
                    st.error("⚠️ Security answer is required.")
                elif pwd != confirm_pwd:
                    st.error("❌ Passwords do not match.")
                else:
                    try:
                        with get_db() as c:
                            c.execute("INSERT INTO users VALUES (NULL, ?, ?, ?, ?, ?)", (uname, email, hash_txt(pwd), sq, hash_txt(sa.lower().strip())))
                        st.session_state.token = make_jwt(email)
                        st.success("✅ Account created!")
                        time.sleep(1)
                        navigate("Dashboard")
                    except sqlite3.IntegrityError:
                        st.error("❌ Email or Username already registered.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Back to Sign In", use_container_width=True): navigate("Login")

        elif st.session_state.page == "Forgot":
            auth_header("Reset your password", "Choose your verification method")

            # ---- Step 0: choose method ----
            if not st.session_state.reset_email:
                email = st.text_input("Registered email address", placeholder="you@infosys.com").lower().strip()
                st.markdown("<br>", unsafe_allow_html=True)

                col_sq, col_otp = st.columns(2)
                if col_sq.button("Via Security Question", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT security_question FROM users WHERE email=?", (email,)).fetchone()
                    if r:
                        st.session_state.reset_email = email
                        st.session_state.sq_p = r[0]
                        st.session_state.reset_mode = "sq"
                        st.rerun()
                    else: st.error("❌ Email not found.")

                if col_otp.button("Via OTP", use_container_width=True):
                    with get_db() as c: r = c.execute("SELECT id FROM users WHERE email=?", (email,)).fetchone()
                    if not r:
                        st.error("❌ Email not found.")
                    else:
                        otp = f"{random.randint(0, 999999):06d}"
                        if send_otp_email(email, otp):
                            st.session_state.otp_code = otp
                            st.session_state.otp_expiry = time.time() + 300  # 5 min
                            st.session_state.otp_sent_to = email
                            st.session_state.reset_email = email
                            st.session_state.reset_mode = "otp"
                            st.success(f"✅ OTP sent to {email}")
                            time.sleep(1)
                            st.rerun()

            # ---- Step 1: security question flow ----
            else:
                if st.session_state.get("reset_mode") == "sq":
                    st.info(f"❓ **Security Question:** {st.session_state.sq_p}")
                    ans = st.text_input("Your answer").lower().strip()
                    npw = st.text_input("New password (min 8 chars)", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.markdown("<br>", unsafe_allow_html=True)
                    if st.button("Reset Password →", use_container_width=True):
                        if len(npw) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: r = c.execute("SELECT security_answer_hash FROM users WHERE email=?", (st.session_state.reset_email,)).fetchone()
                            if r and check_txt(ans, r[0]):
                                with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                                st.success("✅ Password updated successfully!"); time.sleep(1); st.session_state.reset_email = None; navigate("Login")
                            else: st.error("❌ Incorrect security answer.")

                # ---- Step 1b: OTP flow ----
                elif st.session_state.get("reset_mode") == "otp":
                    st.info(f"📧 A 6-digit code was sent to **{st.session_state.otp_sent_to}**. It expires in 5 minutes.")
                    entered_otp = st.text_input("Enter OTP", max_chars=6, placeholder="6-digit code")
                    npw = st.text_input("New password (min 8 chars)", type="password")
                    confirm_npw = st.text_input("Confirm new password", type="password")
                    st.markdown("<br>", unsafe_allow_html=True)

                    col_verify, col_resend = st.columns(2)
                    if col_verify.button("Verify & Reset →", use_container_width=True):
                        if time.time() > (st.session_state.otp_expiry or 0):
                            st.error("❌ OTP expired. Please request a new one.")
                        elif entered_otp != st.session_state.otp_code:
                            st.error("❌ Incorrect OTP.")
                        elif len(npw) < 8:
                            st.error("⚠️ Password must be at least 8 characters long.")
                        elif npw != confirm_npw:
                            st.error("❌ Passwords do not match.")
                        else:
                            with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                            st.success("✅ Password updated successfully!")
                            time.sleep(1)
                            st.session_state.reset_email = None
                            st.session_state.otp_code = None
                            navigate("Login")

                    if col_resend.button("Resend OTP", use_container_width=True):
                        otp = f"{random.randint(0, 999999):06d}"
                        if send_otp_email(st.session_state.reset_email, otp):
                            st.session_state.otp_code = otp
                            st.session_state.otp_expiry = time.time() + 300
                            st.success("✅ New OTP sent.")

            st.markdown("<br>", unsafe_allow_html=True)
            if st.button("← Cancel", use_container_width=True):
                st.session_state.reset_email = None
                st.session_state.reset_mode = None
                st.session_state.otp_code = None
                navigate("Login")

# ============================================================
# DASHBOARDS (ADMIN vs USER)
# ============================================================
else:
    payload = verify_jwt(st.session_state.token)
    if not payload:
        st.session_state.token = None
        st.session_state.page = "Login"
        st.rerun()

    email = payload["email"]
    with get_db() as c: uname = c.execute("SELECT username FROM users WHERE email=?", (email,)).fetchone()[0]

    with st.sidebar:
        st.markdown(f"""
        <div style="padding:20px 8px;text-align:center;">
            <div style="font-size:26px;">🏛️</div>
            <div style="font-weight:700;font-size:16px;font-family:'Playfair Display',serif;color:#ffffff;">Infosys Freight Quote Portal</div>
            <div style="font-size:11px;color:#b7bfd6;letter-spacing:0.5px;">{"ADMIN PANEL" if email=="infosys@ai" else "ENTERPRISE ANALYTICS"}</div>
        </div><hr style="border-color:{COLORS['gold']};opacity:0.3;">
        """, unsafe_allow_html=True)

        opts = ["Dashboard", "Settings", "Logout"] if email=="infosys@ai" else ["Dashboard", "Analytics", "Reports", "Logout"]
        menu = option_menu(None, opts, icons=["house", "gear", "box-arrow-right"] if email=="infosys@ai" else ["house", "graph-up", "file-text", "box-arrow-right"],
                           styles={
                               "container": {"background-color": "transparent"},
                               "icon": {"color": COLORS['gold']},
                               "nav-link": {"color": "#d7dcea", "font-family": "Inter"},
                               "nav-link-selected": {"background-color": COLORS['gold'], "color": COLORS['navy_deep']}
                           })
        if menu == "Logout":
            st.session_state.token = None
            st.session_state.page = "Login"
            st.rerun()

    header_bg = f"linear-gradient(90deg, {COLORS['navy_deep']} 0%, {COLORS['navy']} 100%)"

    if email == "infosys@ai":
        st.markdown(f"""
        <div style="background:{header_bg};border-radius:12px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;border-bottom:3px solid {COLORS['gold']};">
            <div><h1 class="freight-banner-title" style="margin:0;font-size:24px !important;">🏛️ Infosys Freight Quote Portal</h1><div style="color:#b7bfd6;font-size:13px;">Admin Control Panel</div></div>
            <div style="background:{COLORS['gold']};padding:8px 18px;border-radius:6px;font-weight:700;color:{COLORS['navy_deep']};">🛡️ {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        st.markdown(f"""
        <div class="pn-card" style="text-align:center;padding:60px 20px;">
            <h1 style="font-size:32px !important;margin-bottom:10px;">🛡️ Admin Dashboard</h1>
            <p style="color:{COLORS['text_muted']};font-size:16px;font-weight:500;">Welcome to the Administrator area.</p>
        </div>
        """, unsafe_allow_html=True)

    else:
        st.markdown(f"""
        <div style="background:{header_bg};border-radius:12px;padding:24px 32px;display:flex;justify-content:space-between;align-items:center;margin-bottom:24px;border-bottom:3px solid {COLORS['gold']};">
            <div><h1 class="freight-banner-title" style="margin:0;font-size:24px !important;">🏛️ Infosys Freight Quote Portal</h1><div style="color:#b7bfd6;font-size:13px;">Enterprise Analytics Dashboard</div></div>
            <div style="background:{COLORS['gold']};padding:8px 18px;border-radius:6px;font-weight:700;color:{COLORS['navy_deep']};">👤 {uname}</div>
        </div>
        """, unsafe_allow_html=True)

        c1, c2, c3, c4 = st.columns(4)
        for col, icon, lbl, val in [(c1, "📄", "Documents Indexed", "128"), (c2, "🔍", "Searches Today", "47"),
                                    (c3, "📊", "Efficiency Score", "98.4%"), (c4, "🛡️", "Security Status", "Secured")]:
            col.markdown(f"""
            <div class="pn-card" style="text-align:center;">
                <div style="font-size:26px;">{icon}</div>
                <div style="font-size:24px;font-weight:700;color:{COLORS['text_heading']};font-family:'Playfair Display',serif;">{val}</div>
                <div style="color:{COLORS['text_muted']};font-size:12px;font-weight:600;letter-spacing:0.3px;">{lbl}</div>
            </div>
            """, unsafe_allow_html=True)

        st.markdown("<br>", unsafe_allow_html=True)
        fig = go.Figure(go.Indicator(mode="gauge+number", value=92, title={"text": "System Health Index", "font": {"color": COLORS['text_heading'], "size": 14}},
                        gauge={"axis": {"range": [0, 100]}, "bar": {"color": COLORS['gold']}, "bgcolor": COLORS['bg_card'], "borderwidth": 1, "bordercolor": COLORS['navy']}))
        fig.update_layout(paper_bgcolor="rgba(0,0,0,0)", font={"color": COLORS['text_main'], "family": "Inter"}, height=260, margin=dict(l=10, r=10, t=40, b=10))
        st.plotly_chart(fig, use_container_width=True)

Overwriting app.py


In [22]:
import os
import time
import subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Retrieve your secret token securely from Colab Secrets
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Kill any existing ngrok tunnels or streamlit sessions
ngrok.kill()
!pkill -f streamlit

# 3. Start Streamlit in the background on port 8501
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# 4. Open ngrok tunnel
public_url = ngrok.connect(8501).public_url
print("=" * 60)
print(f"🚀 Infosys Portal Live URL: {public_url}")
print("=" * 60)
print("⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.")

try:
    # Keep the cell active so Ctrl+C can be intercepted
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print("\n" + "🛑" * 30)
    print("Received Ctrl+C / Stop signal. Shutting down...")
    ngrok.kill()
    process.terminate()
    !pkill -f streamlit
    print("✅ Ngrok tunnel closed and Streamlit server stopped gracefully.")


🚀 Infosys Portal Live URL: https://snitch-disperser-sterling.ngrok-free.dev
⏳ App is running! Press [Ctrl + C] or the Colab Stop button to shut down.

🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑🛑
Received Ctrl+C / Stop signal. Shutting down...
✅ Ngrok tunnel closed and Streamlit server stopped gracefully.
